# Lecture 7 · Notebook 2 — pass@k vs. pass^k: trying more, and being reliable
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NLP-Course-CUHKSZ/NLP-course-cuhksz.github.io/blob/main/materials/tutorials/lecture07/lecture07_2_pass_at_k_vs_pass_hat_k.ipynb)

**Companion to [Chapter 8 of the video lecture](https://nlp-course-cuhksz.github.io/lecture07.html#p8-s125)** (slide 125, *Graders, and why trying more helps*) and to
principle 9 on [slide 131](https://nlp-course-cuhksz.github.io/lecture07.html#p8-s131): *measure pass^k, the reliability users experience*. An agent that solves a task 3 times out
of 4 succeeds with probability 98% if it may try three times and keep a run the grader accepts, but all three runs succeed only 42% of
the time. In this notebook you will:

1. reproduce those two numbers and plot both metrics against k,
2. estimate pass@k and pass^k **from a finite number of trials**, with the standard unbiased estimators,
3. see two agents with the **same pass@1** but very different reliability,
4. see why "try more" only helps if the **grader** can tell which try succeeded, and
5. check whether a harness change really helped, with a significance test.

Pure NumPy, no model, no GPU: the whole notebook runs in under a minute.

In [ ]:
import numpy as np, matplotlib.pyplot as plt
from math import comb
rng = np.random.default_rng(0)

## 1 · One task, success probability p
If each trial succeeds independently with probability p:

* **pass@k** = P(at least one of k trials succeeds) = 1 − (1 − p)^k, the payoff of trying more (with a grader that picks the winner),
* **pass^k** = P(all k trials succeed) = p^k, how often a user who runs the agent k times is never let down.

In [ ]:
p = 0.75
for k in [1, 2, 3, 5, 10]:
    print(f"k={k:2d}   pass@k = {1 - (1 - p) ** k:6.1%}   pass^k = {p ** k:6.1%}")

ks = np.arange(1, 11)
plt.figure(figsize=(6, 3.5))
for p in [0.5, 0.75, 0.9]:
    plt.plot(ks, 1 - (1 - p) ** ks, "-o", ms=3, label=f"pass@k, p={p}")
    plt.plot(ks, p ** ks, "--o", ms=3, label=f"pass^k, p={p}")
plt.xlabel("k (number of trials)"); plt.ylabel("probability"); plt.legend(fontsize=7, ncol=2); plt.grid(alpha=.3)
plt.title("Same agent, two questions"); plt.show()

## 2 · Estimating both metrics from n trials
On a benchmark we do not know p. We run each task n times and count c successes. The unbiased estimators are

* pass@k ≈ 1 − C(n−c, k) / C(n, k)   (Chen et al., 2021, the Codex paper),
* pass^k ≈ C(c, k) / C(n, k)   (Yao et al., 2024, τ-bench),

averaged over tasks. They use all n trials, which is less noisy than looking at only the first k. Below, a simulated agent on 100
tasks whose difficulty varies: per-task success probabilities come from a Beta distribution. The estimates differ from the true values
only by sampling noise, which grows as k approaches n.

In [ ]:
def pass_at_k(n, c, k):
    return 1.0 if n - c < k else 1.0 - comb(n - c, k) / comb(n, k)

def pass_hat_k(n, c, k):
    return comb(c, k) / comb(n, k)

T, n = 100, 10
p_task = rng.beta(2.0, 1.0, size=T)                 # mean success 0.67, some tasks hard, many easy
c = rng.binomial(n, p_task)                          # successes out of n trials per task

print(" k   pass@k true  est.    pass^k true  est.")
for k in [1, 2, 4, 8]:
    true_at, true_hat = np.mean(1 - (1 - p_task) ** k), np.mean(p_task ** k)
    est_at = np.mean([pass_at_k(n, ci, k) for ci in c])
    est_hat = np.mean([pass_hat_k(n, ci, k) for ci in c])
    print(f"{k:2d}   {true_at:9.3f}  {est_at:.3f}   {true_hat:9.3f}  {est_hat:.3f}")

## 3 · Same pass@1, different agents
Two agents both solve 60% of trials. The **consistent** agent always solves 60 of the 100 tasks and never solves the other 40.
The **flaky** agent solves every task 60% of the time. A single-run leaderboard cannot tell them apart.

In [ ]:
consistent = np.r_[np.ones(60), np.zeros(40)]
flaky = np.full(100, 0.6)
ks = np.arange(1, 11)
fig, ax = plt.subplots(1, 2, figsize=(10, 3.5), sharey=True)
for name, pt in [("consistent", consistent), ("flaky", flaky)]:
    ax[0].plot(ks, [np.mean(1 - (1 - pt) ** k) for k in ks], "-o", ms=3, label=name)
    ax[1].plot(ks, [np.mean(pt ** k) for k in ks], "-o", ms=3, label=name)
ax[0].set_title("pass@k: trying more"); ax[1].set_title("pass^k: reliability")
for a in ax: a.set_xlabel("k"); a.grid(alpha=.3); a.legend()
ax[0].set_ylabel("score"); plt.show()
print("pass@1: consistent", consistent.mean(), "| flaky", flaky.mean())

With retries and a perfect grader, the flaky agent looks far better (pass@10 ≈ 1.0 vs 0.6). For a user who needs the job done every
time, it is far worse (pass^10 ≈ 0.006 vs 0.6). Which one you want depends on whether a grader stands between the agent and the user.

## 4 · Trying more needs a grader
Slide 125 ends with a catch: *trying more only helps if a grader can tell which try succeeded*. Model it: each try succeeds with
probability p. The grader accepts a correct run with probability `tpr` and **wrongly** accepts a failed run with probability `fpr`.
The harness runs up to k tries and returns the first accepted run (or the last run if none is accepted).

In [ ]:
def best_of_k(p, k, tpr, fpr, trials=200_000):
    ok = rng.random((trials, k)) < p
    accepted = np.where(ok, rng.random((trials, k)) < tpr, rng.random((trials, k)) < fpr)
    first = np.where(accepted.any(1), accepted.argmax(1), k - 1)
    return ok[np.arange(trials), first].mean()

p, ks = 0.5, np.arange(1, 9)
plt.figure(figsize=(6, 3.5))
plt.plot(ks, 1 - (1 - p) ** ks, "k--", label="oracle grader (= pass@k)")
for tpr, fpr in [(0.95, 0.0), (0.95, 0.1), (0.9, 0.3), (0.8, 0.6)]:
    plt.plot(ks, [best_of_k(p, k, tpr, fpr) for k in ks], "-o", ms=3, label=f"tpr={tpr}, fpr={fpr}")
plt.xlabel("k (tries allowed)"); plt.ylabel("success delivered to the user"); plt.grid(alpha=.3); plt.legend(fontsize=8)
plt.title(f"Best-of-k with an imperfect grader (p = {p})"); plt.show()

A grader that lets failures through puts a ceiling on what retries can buy: success saturates at
p·tpr / (p·tpr + (1−p)·fpr), the grader's precision, no matter how large k gets. That is why the lecture spends a slide on code-based,
model-based and human graders, and why *grade the outcome, not the agent's claim* (slide 124).

## 5 · Did the harness change really help?
The steering loop on slide 94: before the fix, the agent said "done" with failing tests in 6 of 20 tasks; after adding a test hook, 0 of
20. And on slide 80, SWE-agent's interface fixes 18% of 300 issues against 11% for a plain shell, with the same GPT-4 Turbo. Are those
differences bigger than chance? Fisher's exact test answers that for two success counts.

In [ ]:
from scipy.stats import fisher_exact

def compare(name, a_succ, a_n, b_succ, b_n):
    _, pval = fisher_exact([[a_succ, a_n - a_succ], [b_succ, b_n - b_succ]])
    print(f"{name:42s} {a_succ}/{a_n} vs {b_succ}/{b_n}   p = {pval:.4f}")

compare("false 'done' before vs after the hook (s94)", 6, 20, 0, 20)
compare("plain shell vs SWE-agent interface (s80)", 33, 300, 54, 300)
compare("15 runs per harness: 7 vs 11 successes", 7, 15, 11, 15)

The first two differences are unlikely to be chance (p < 0.05). The third, the size of the experiment in Notebook 1, is not: with 15
runs per harness, a 27-point gap can still be noise. Trials within the same task are also correlated, so for real claims run more tasks,
not only more trials per task.

## Exercises
1. **How many trials?** In section 2, set `n = 3` and estimate pass^8. What goes wrong, and why does the estimator need n ≥ k?
2. **A cheaper grader.** In section 4, a model-based grader has `tpr=0.9, fpr=0.2` and costs 1 unit per try; a human costs 20 units with
   `tpr=0.99, fpr=0.01`. For p = 0.4, which gives more success per unit of cost at k = 4?
3. **Your own agent.** Paste the `res` table from Notebook 1 (a tasks × trials 0/1 array) and compute pass@k and pass^k with the
   estimators from section 2 for k = 1…5.

**Back:** [Notebook 1 — a ReAct agent loop]({colab(N1)}) · **Next:** [Notebook 3 — agent memory: files vs. embeddings]({colab(N3)})